In [2]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=r9cwTOrha9KWgfNVzQeX4qtShI2sBq&access_type=offline&code_challenge=a8MZLw573vplVj8iSa_CjYttn-HsNppiEyxPk-C-s98&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [3]:
import asyncio
from google import genai
from google.genai import types
import os
# ---------- Setup ----------

client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# ---------- Helper ----------

def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = path.split(".")[-1].lower()
    mime = {
        "pdf": "application/pdf",
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "txt": "text/plain",
        "json": "application/json"
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)
# ---------- Output Saving ----------







In [5]:
def build_prompts():
    return {
        "CODE_LOOKUP": """You are a building code compliance analyst preparing an expert report from the attached carrier estimate or supplemental documents.

OBJECTIVE:
Extract location-specific property information and generate a code compliance matrix to assess the validity of the restoration scope. Your matrix must allow technical, legal, and enforcement-level review.

PART 1: PROPERTY IDENTIFICATION

Extract and display the following details explicitly:

| Field | Value | Source |
|-------|-------|--------|
| Full Street Address | [123 Main St, Anytown, TX 75001] | [Page #, Document Name] |
| Municipality or Jurisdiction | [City of Anytown] | [zoning info, doc line #] |
| County | [Dallas County] | [tax record or doc] |
| Incorporated/Unincorporated | [Incorporated] | [jurisdictional map or site] |
| ZIP Code | [75001] | [carrier estimate] |
| Inspection/Report Date | [March 15, 2024] | [carrier estimate or inspection] |

MUST include source citations in all rows.

PART 2: CODE STACK DETERMINATION

Using the jurisdiction and inspection date, determine the full set of codes applicable at the time of inspection. List each code family separately:

| Code Type | Version/Edition | Citation Source | Applied Amendments | Enforceability Level |
|-----------|------------------|------------------|---------------------|------------------------|
| IRC | 2018 IRC | [ICC database, city site] | [NCTCOG mods] | Mandatory |
| NEC | 2023 NEC | [NEC.gov] | [none] | Mandatory |
| IECC | 2021 IECC | [DOE/state site] | [state energy code mods] | Mandatory |

For each code:
- Confirm it applies to residential work
- Cross-verify adoption at municipal, county, and state levels
- Include any stricter local amendments or enforcement practices

Do not skip any level. If any level lacks information, state "Unknown" and flag it.

PART 3: CODE COMPLIANCE MATRIX

Build a matrix of building components and related code requirements, grounded in enforceable citations and carrier estimate inclusion review.

Matrix Format:

| Affected System | Code Section | Code Summary | Interpretation | Required By Code | Present in Carrier Estimate | Justification |
|------------------|---------------|----------------|------------------|-------------------|-------------------------------|----------------|
| Roofing | IRC R905.2.8.5 | Drip edge required at eaves & rakes | Must be installed per manufacturer & code | Yes | No | Missing on eaves; required due to full shingle tear-off |

Mandatory Coverage Areas (review all):
- Roofing: decking inspection, underlayment, ice/water shield, flashing, drip edge, ventilation, fasteners
- Siding: WRB, flashing, trim
- Windows: flashing, insulation, support
- Framing: blocking, shear, load path
- Electrical: grounding, junctions, disconnects
- HVAC: clearances, lines, platforms, ductwork

For each system:
- Include at least one row per bullet above unless clearly not applicable
- Use code section numbers (e.g., IRC R806.2)
- Justify “Not Required” with exact wording from code or trigger logic
- List at least one field as “No” under ‘Present in Carrier Estimate’ for auditing

VALIDATION AND QUALITY CHECKS

Code Version Accuracy:
- Confirm all adopted code versions match inspection/report date

Trigger Identification:
- Every cited requirement must identify the action that triggers it (e.g., roof tear-off)

Estimate Comparison:
- Flag all omissions; justify any marked "Yes" with estimate page or line reference

“N/A” Handling:
- Only use “Not Applicable” when justified by jurisdictional exemption or clear logic

Jurisdiction Checks:
- Verify amendments using official municipal, county, and state websites
- Cite the source used for each code adoption decision

Matrix Completeness:
- If a system has no entries, explain why
- Include “Inspection Required” for components not visibly verifiable

DAUBERT RELIABILITY REQUIREMENTS

Your analysis must meet legal admissibility standards:

- Reliability: All citations must be verified through ICC, NEC, or government sources
- Known Error Rate: Flag interpretations that involve ambiguity or enforcement discretion
- Peer Review: Reference ICC, AIA, NCARB or equivalent professional publications
- General Acceptance: Ensure all recommendations reflect standard industry enforcement

FINAL OUTPUT FORMAT

Provide the following 3 sections in order:

1. Property Details Table
2. Code Adoption Table
3. Code Compliance Matrix Table

Use markdown-compatible table formatting if supported. Do not summarize or paraphrase—this is a structured code compliance report.
""",

        "REPORT_ANALYSIS": """You are a forensic damage analyst specializing in residential and commercial property insurance claims. You are provided with one or more of the following document types as attachments:

* Forensic inspection reports (e.g., roof, interior, structural)
* Annotated images or photo sets
* Engineering letters or reports
* Aerial roof measurement data (e.g., EagleView or similar)
* Claim summaries, contractor notes, and carrier communications

NOTE: The file names may vary and may not explicitly state their contents. Do not rely on filenames. Instead, identify each document type by its internal content (e.g., “roof slope diagrams,” “photo with date/time metadata,” “structural integrity opinion,” etc.).

OBJECTIVE:
Perform a complete, evidence-based analysis of all observable damages by room or elevation. For every observed condition, extract supporting documentation and generate a quantifiable, scoped breakdown tied directly to measurable evidence.

DO NOT SKIP ANY STEP IF A DOCUMENT TYPE IS MISSING. Use only what is available and note omissions where appropriate. Proceed regardless of gaps in input.

STRUCTURE:
Repeat the following format for **every room, elevation, or system area** with documented or inferable damage.

---

### [Room or Elevation Name]

DAMAGE DOCUMENTATION

* **Primary Damage**: Describe the main damage (e.g., water stain, blistering, rot, delamination)
* **Secondary Damage**: Any follow-on effects (e.g., mold, insulation compromise, trim swelling)
* **Evidence Sources**: Reference Photos [IDs or filenames], Report pages [#], Inspection Notes, and don't forget to mention the document you got it from.

QUANTIFICATION MATRIX

| Element  | Damaged Area | Unit | Measurement Method        | Evidence Source         |
| -------- | ------------ | ---- | ------------------------- | ----------------------- |
| Ceiling  | [#]         | SF   | [e.g., stain boundaries] | Photo [#], Report [#] |
| Walls    | [#]         | SF   | [height x length]        | Photo [#], Report [#] |
| Trim     | [#]         | LF   | [linear measurement]     | Photo [#], Report [#] |
| Fixtures | [#]         | EA   | [count visible items]    | Photo [#]              |

CARRIER ESTIMATE COMPARISON

* **Included Scope**: List what the carrier did include (line item description)
* **Quantity Variance**: Carrier [#] vs. Observed [#] ([percentage variance])
* **Missing Scope**: Items observed but omitted in carrier scope
* **Justification for Inclusion**: Why the omitted item is required (trigger logic, industry standard, interdependent component)

SYSTEM INTEGRATION IMPACTS

* **Code Triggers**: Any work that initiates building code compliance requirements (e.g., R-value updates, decking inspections)
* **Aesthetic Impacts**: Line-of-sight disruptions, matching failures (color, texture, material)
* **Sequence Dependencies**: Where partial work is infeasible due to construction sequence (e.g., finish ceiling must follow after insulation)

RISK ASSESSMENT

* **Immediate Concerns**: Mold, electrical hazard, structural weakness, open exposure
* **Long-Term Implications**: Warranty issues, insulation failure, degraded performance, continued leakage
* **Mitigation Requirements**: Any required steps to prevent escalation (e.g., full replacement, drying protocol, temporary protection)

---

EVIDENCE CORRELATION GUIDELINES

For **every damage condition**, you must:

* Link to at least one **photo ID** or annotated image or  logical reasoning from analysis
* Cite page number or section from the relevant inspection, engineering, or aerial report
* If quantification is inferred (e.g., measurement from image scale), specify the method used
* Do not make undocumented assumptions; flag any gaps explicitly

ALWAYS USE language like:

* "As shown in Photo 14, ceiling discoloration extends 7 feet from the entry"
* "Page 3 of Engineering Report notes compromised rafter tail at southwest corner"

MANDATORY OUTPUT REQUIREMENTS

1. You must output one full section per distinct room/elevation/system
2. Do not omit any damaged item or area, even small trim or ceiling tape lines
3. Tie every observed damage to at least one form of verifiable evidence
4. If no damage is observed in an area, write:
   "**No observable damage**: This room showed no visible damage based on available documents and photos. No scope required."

VALIDATION PROTOCOLS

* Every damage claim must have at least one evidence reference
* All quantities must be measurable or reasonably inferred
* No item should be included without a supporting cause
* "Not observed" is acceptable only with explanation (e.g., area not visible in photos)

QUALITY ASSURANCE CHECKLIST

* Verify all room/elevation names match across report and photo metadata
* Ensure unit types are consistent across matrix (SF, LF, EA)
* Cross-check all photo references are unique and traceable
* Check scope logic for carrier comparisons: line items and units must align

ALLOWED ASSUMPTIONS

If any document type is missing:

* Use remaining evidence to fill gaps
* Infer plausible scope based on visible consequences
* Flag assumptions clearly as "inferred from photo only" or "no direct engineering note found"

You must **never halt or skip analysis due to missing files**.

CONCLUSION

Complete the output for all rooms or elevations with observed damage. Do not stop at major rooms—include small closets, utility areas, and transitions. Every component matters.
"""
,

        "SCOPING_LOGIC": """You are a restoration estimator building a full plaintiff-style scoping matrix. Your job is to ensure every valid line item is captured per code, damage, and standard practices.

**OBJECTIVE**: Generate complete scope justification covering all valid restoration requirements.

Structure your scope by the following **six domains**, and under each, break down **component-by-component**.

---

### **1. Code-Driven Requirements**
For each component (roofing, siding, electrical, HVAC, etc.), include:

- **Code Requirement (w/ citation)**  
- **What triggers the requirement** (e.g., tear-off, disturbed assembly)  
- **Expected Line Items** (demo + install)  
- **Consequences of omission** (warranty void, leaks, mold)

Example:
- **Roof Ventilation (IRC R806.2)**: If shingles are removed, intake/exhaust balance must be verified; turtle vents or ridge vent required if not already compliant.

---

### **2. Mandatory Scope Inclusions**
These items are required based on **scope sequencing**, not visible damage.

**Process**:
1. **Map restoration sequence** (demo → rough → finish)
2. **Identify unavoidable impacts** (what gets disturbed)
3. **Specify replacement requirements** (what can't be reused)
4. **Document industry standards** (manufacturer specs, trade practices)

**Standard inclusions**:
- **Insulation**: [removal/replacement triggers]
- **Vapor barriers**: [damage during demo/install]
- **Pipe jacks**: [single-use items requiring replacement]
- **Fixture detachment**: [temporary removal requirements]
- **System disconnections**: [HVAC, electrical safety requirements]

**Justification format**:
- **Item**: [specific scope component]
- **Trigger**: [why it's unavoidable]
- **Standard**: [industry/manufacturer requirement]
- **Cost of omission**: [failure consequence]

---

### **3. Matching, Aesthetic, and LKQ Rules**
Explain when partial replacement is inappropriate:

- Define visual mismatch triggers: color, sheen, exposure age
- Mention **line-of-sight logic** (e.g., hallway ceiling vs. bedroom)
- Detail material availability issues (discontinued trim, aged siding)
- Explain “paint from corner to corner” rule
- Apply these to:
    - Shingles
    - Siding
    - Trim & base
    - Interior ceilings

---

### **4. Site Protection & Containment**
List materials and labor needed to protect the site, for each major work area.

- Floor covering (Ram board, poly)
- Dust containment (zip walls, negative air)
- HEPA air scrubbers (during drywall demo or mold remediation)
- Debris management
- Furniture moving or content manipulation (by room)

**Justification criteria**:
- **Property value protection**: [preventing additional damage]
- **Health and safety**: [dust, debris, contaminant control]
- **Code compliance**: [required protection standards]
- **Warranty requirements**: [manufacturer specifications]
---

### **5. General Conditions & Overhead**
Define what GC-level provisions are triggered:

- Project manager (daily hours, scheduling)
- Dumpster, job toilet, material storage (quantified)
- Permits (when and why needed)
- State/local sales tax inclusion
- O&P (applied if ≥3 trades OR complex coordination)

Justify each with reasoning: “Required due to 4+ trade interaction in confined space,” etc.

---

### **6. Paint & Finish Standards**
Explain proper finish sequencing:

- New drywall: 1 primer + 2 finish coats minimum
- Ceilings: must be painted full-plane to match sheen
- Blending: describe when wall-to-wall blending is required
- Texture matching: (e.g., knockdown vs smooth Level 4)

**VALIDATION PROTOCOL**:
- Every scope item must have clear justification
- All code citations must be current and accurate
- Aesthetic standards must be objectively measurable
- Sequence logic must be technically sound
- Protection requirements must be quantifiable
- Overhead must be proportional to project complexity

""",

        "ESTIMATE": """You are a certified insurance restoration estimator creating plaintiff-style cost breakdowns using Xactimate methodology.

**OBJECTIVE**: Generate mathematically precise, fully justified estimates with complete cost calculations.

You are required to apply strict pricing logic for every line item using the following structure. All calculations must be mathematically exact. Do not round totals prematurely.
Mandatory Calculation Rules
1. Direct Cost (DC)
DC = QTY × UNIT PRICE

Ensure correct units (e.g., SF, LF, EA) are applied
Never average across multiple items—each scope must have a distinct and accurate QTY and UNIT PRICE

2. Material Sales Tax (TAX)

TAX applies only to the material portion of each line item
The material_tax_rate is provided in the MASTER_INPUT_DATA (e.g., 8.75%)
If the line item contains both labor and material, apply tax proportionally to the material portion only
For pure labor items, TAX = $0.00

3. Overhead & Profit (O&P)

Apply 10% Overhead + 10% Profit (compounded) to the subtotal of (Direct Cost + TAX)
O&P = (DC + TAX) × 0.20

4. Replacement Cost Value (RCV)

RCV = DC + TAX + O&P
This value must match exactly with the calculated sum per line item

5. Depreciation (DEPREC.)

For this estimate, DEPREC. = $0.00 unless explicitly instructed otherwise
ACV = RCV - DEPREC.

6. Precision Mandate

Do not round values until final output; retain at least 2 decimal places
All subtotal and grand total calculations must match the sum of their respective line items exactly
Never display or include a line item without completing all columns

Mandatory Output Format
For each room or area, provide a markdown-formatted table enclosed in a code block, using the following columns in this exact order:
CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV
Requirements:

Each item must include valid Xactimate CAT and SEL codes
Accurate QTYs and pricing consistent with localized labor/material rates
Do not use placeholders
Ensure the subtotal for each Room/Area is provided after its table
Followed by a section total
Finally a Grand Total Summary combining all rooms and the Addendum section if applicable

MANDATORY PRICING SOURCE & PRICE LIST USAGE:

• From the provided carrier estimate, extract the **full property address** including ZIP code and state.
• Then retrieve the appropriate localized pricing dataset using **Xactimate-style price list codes**.
• If no price list code is given, dynamically infer the correct regional code using the ZIP code from the address and retrieve publicly available construction cost data.


Your estimate must be grouped by **room or elevation**, and for each group, output:

---

### **[Room or Elevation Name]**

| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV |
|-----|-----|-------------|-----|------|------------|-----|-----|-----|----------|-----|

- **Line-by-line reasoning**: After each table, briefly explain why each major item was included (e.g., “Blistered ceiling, Report pg 4,” “Drip edge missing, IRC R905.2.8.5”)
- **[Description]**: [damage reference, code requirement, or scope logic]
- **[Description]**: [evidence source, photo reference, or standard practice]

---

After all areas, include:

---

### **GENERAL CONDITIONS**

| DESCRIPTION | QTY | UNIT | UNIT PRICE | TOTAL |
|-------------|-----|------|------------|--------|
| Project Supervision | XX hrs | HR | $ | $ |
| Dumpster | 1 | EA | $ | $ |
| Portable Toilet | 1 | EA | $ | $ |
| Permit Cost | 1 | EA | $ | $ |

Explain each line in a brief bullet list: “Permit required for electrical disconnect per city ordinance.”

---

### **GRAND TOTALS**

| Category | Subtotal |
|----------|----------|
| Roofing | $X |
| Siding/Exterior | $Y |
| Interior | $Z |
| General Conditions | $W |
| O&P (10% + 10percent) | $V |
| **Grand Total (RCV)** | **$[final total]** |

---


---

### **Aesthetic Restoration Addendum**
Summarize any full-system replacements (roof, siding, ceiling) that were required due to:
- Inability to match
- Material brittleness
- Line-of-sight
- Manufacturer discontinuation

**XACTIMATE CODE REQUIREMENTS**:
- **CAT codes must be valid Xactimate categories**:
  - DRY (Drywall), ROF (Roofing), SID (Siding), PAI (Painting), INS (Insulation)
  - DEM (Demolition), ELE (Electrical), PLB (Plumbing), HVA (HVAC), FLR (Flooring)
  - WIN (Windows), CAB (Cabinetry), TRI (Trim), CEI (Ceiling), WAL (Walls)
  - APD (Applied/Adhesive), MLD (Mold), CON (Concrete), CAP (Carpentry)

- **SEL codes must be valid Xactimate selectors**:
  - Use proper base codes (3-4 letters) with correct suffixes
  - Examples: DRY (install drywall), DRYR (repair drywall), DRYP (patch drywall)
  - ROF (roof general), ROFS (roof shingles), ROFU (roof underlayment)
  - PAI (paint general), PAIC (paint ceiling), PAIW (paint walls)

- **Unit codes must match Xactimate standards**:
  - SF (square feet), LF (linear feet), EA (each), HR (hour), SY (square yard)
  - CY (cubic yard), MBF (thousand board feet), GAL (gallon), LB (pound)

Use markdown bullets and include references to photos, reports, or industry rules.
""",

        "REBUTTAL": """You are a forensic rebuttal specialist responding to a deficient insurance carrier estimate. Your response must be formal, detailed, and based in code, evidence, and industry logic.

**OBJECTIVE**: Create comprehensive, defensible rebuttal documentation with legal and technical precision.

### **I. Summary of Discrepancies**
Categorize the major classes of omissions (e.g., code compliance, aesthetic mismatch, missing scope) with high-level bullets.

---

### **II. Room-by-Room Rebuttal**

#### [Room or Elevation Name]
- **Issue:** What was omitted or under-scoped
- **Evidence:** Photo X, Report pg Y
- **Code/Standard:** IRC section, IICRC standard, or Xactimate convention
- **Correct Scope:** Describe what should be included
- **Reasoning:** Include logic based on damage extent, mismatch, sequence of construction

---

### **III. Code Violations**
List every component omitted or under-scoped that violates building code:
- IRC R908.3.1: Decking not allowed to remain without inspection
- NEC 820.100: Satellite system ungrounded

---

### **IV. General Conditions & O&P Justification**
- Number of trades
- Need for project supervision
- Dumpster/toilet/storage logic
- Code-permitted markup (O&P)

---

### **V. Aesthetic & Matching Justifications**
- Why patching fails LKQ standard
- Photo-based mismatch documentation
- Manufacturer unavailability (if applicable)

---

### **VI. Conclusion**
Summarize:
- # of omitted rooms or trades
- Major life-safety risks or code issues
- Estimated value delta (if known)
- Your demand: “We respectfully request that the omitted items be added and paid in full.”

**VALIDATION CHECKLIST**:
- [ ] Every deficiency has supporting evidence
- [ ] All code citations are current and accurate
- [ ] Financial calculations are mathematically correct
- [ ] Professional tone maintained throughout
- [ ] Specific actions requested clearly stated
- [ ] Documentation references complete and accurate

Maintain a clear, professional tone rooted in documentation.
"""
    }


In [12]:

def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2500008/run2"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")

In [6]:
# ---------- Async Gemini Runner ----------

async def run_block(label, prompt, file_parts=None):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    if file_parts:
        contents[0].parts.extend(file_parts)

    output = ""
    try:
        print(f"🔹 Running {label}...")
        stream = client.models.generate_content_stream(
            model=model_name,
            contents=contents,
            config=generate_content_config
        )
        for chunk in stream:  # ✅ DO NOT use 'await'
            output += chunk.text
        print(f"✅ {label} complete ({len(output)} chars)")
    except Exception as e:
        output = f"[ERROR in {label}] {e}"
        print(output)

    return label, output




# ---------- Master Pipeline ----------

async def run_aistimate_pipeline(file_paths):
    prompts = build_prompts()

    # Assign files
    carrier_parts = [make_part(file_paths[0])]
    evidence_parts = [make_part(path) for path in file_paths[0:]]

    # Stage 1: Run code lookup & damage analysis in parallel
    stage1_tasks = [
        run_block("CODE_LOOKUP", prompts["CODE_LOOKUP"], carrier_parts),
        run_block("REPORT_ANALYSIS", prompts["REPORT_ANALYSIS"], evidence_parts),
    ]
    stage1_results = await asyncio.gather(*stage1_tasks)
    context = {label: output for label, output in stage1_results}

    for label, content in stage1_results:
        save_output(label, content)

    # Stage 2: Scoping logic (needs prior outputs)
    scoping_context = (
        f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
    )
    label, scoping_output = await run_block("SCOPING_LOGIC", prompts["SCOPING_LOGIC"] + "nn" + scoping_context)
    save_output(label, scoping_output)
    context["SCOPING_LOGIC"] = scoping_output

    # Stage 3: Estimate generation
    estimate_context = (
        f"--- CODE MANDATES ---n{context['CODE_LOOKUP']}nn"
        f"--- DAMAGE FINDINGS ---n{context['REPORT_ANALYSIS']}nn"
        f"--- SCOPING RULES ---n{context['SCOPING_LOGIC']}"
    )
    label, estimate_output = await run_block("ESTIMATE", prompts["ESTIMATE"] + "nn" + estimate_context)
    save_output(label, estimate_output)

    # Stage 4: Rebuttal
    # Stage 4: Rebuttal (pass carrier file for comparison)
    label, rebuttal_output = await run_block(
    "REBUTTAL",
    prompts["REBUTTAL"] + "nn" + estimate_output,
    file_parts=carrier_parts  # 🔹 passes carrier estimate as input context
    )
    save_output(label, rebuttal_output)


    return {
        "code_lookup": context["CODE_LOOKUP"],
        "report_analysis": context["REPORT_ANALYSIS"],
        "scoping_logic": context["SCOPING_LOGIC"],
        "estimate_output": estimate_output,
        "rebuttal_output": rebuttal_output,
    }


In [7]:

def save_output(label: str, content: str):
    output_dir = f"outputs/Jul16/2500074/run4"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Carrier Estimate ($16,113.56) Insurance Carrier Estimate.pdf",                         # file_paths[0]
    "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Eagleview Report - Grace Forensic Plaintiff Expert Estimate.PDF",                        # evidence
    "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Forensic Damage Assessment - Grace Forensic Plaintiff Expert Estimate.pdf",               # evidence
    "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Inspection Report For Primary Structure 04-16-2025 Plaintiff Expert Estimate_compressed.pdf"                     # evidence
])


🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (7009 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (16626 chars)
📝 Saved: outputs/Jul16/2500074/run4/output_code_lookup.txt
📝 Saved: outputs/Jul16/2500074/run4/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (12730 chars)
📝 Saved: outputs/Jul16/2500074/run4/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (13670 chars)
📝 Saved: outputs/Jul16/2500074/run4/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (12166 chars)
📝 Saved: outputs/Jul16/2500074/run4/output_rebuttal.txt


In [4]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2500008/run5"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Estimate - North American Public Adjusters ($44,60 Plaintiff Expert Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Grace Forensic Photos and Damage Report Plaintiff Expert Estimate_compressed.pdf"
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (7100 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (10707 chars)
📝 Saved: outputs/Jul14/2500008/run5/output_code_lookup.txt
📝 Saved: outputs/Jul14/2500008/run5/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (12665 chars)
📝 Saved: outputs/Jul14/2500008/run5/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (11552 chars)
📝 Saved: outputs/Jul14/2500008/run5/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (11174 chars)
📝 Saved: outputs/Jul14/2500008/run5/output_rebuttal.txt


In [16]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul15/2500005/run7"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/CaseDocuments - 2500005 - Zimmerman v. Homesite Insurance Company 20250606173055/2500005  Carrier Estimate ($20,937.22) Insurance Carrier Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500005 - Zimmerman v. Homesite Insurance Company 20250606173055/2500005 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate_compressed.pdf",
    "Aiestimate/CaseDocuments - 2500005 - Zimmerman v. Homesite Insurance Company 20250606173055/2500005_Plaintiff Insurance Co._Insurance Policy_From_Storm Law Partners_ZIMMERMAN_CAREY_SEPT 2024 NAPA. LONGORIA Esitmate384 copy.pdf"
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (9236 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (39524 chars)
📝 Saved: outputs/Jul15/2500005/run7/output_code_lookup.txt
📝 Saved: outputs/Jul15/2500005/run7/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (18499 chars)
📝 Saved: outputs/Jul15/2500005/run7/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (13335 chars)
📝 Saved: outputs/Jul15/2500005/run7/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (24949 chars)
📝 Saved: outputs/Jul15/2500005/run7/output_rebuttal.txt


In [8]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2550002/run2"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf"
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (6881 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (7297 chars)
📝 Saved: outputs/Jul14/2550002/run2/output_code_lookup.txt
📝 Saved: outputs/Jul14/2550002/run2/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (13461 chars)
📝 Saved: outputs/Jul14/2550002/run2/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (9826 chars)
📝 Saved: outputs/Jul14/2550002/run2/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10248 chars)
📝 Saved: outputs/Jul14/2550002/run2/output_rebuttal.txt


In [ ]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul16/2500083/run5"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/Case File Exports/2550002_Rowlan v. Allstate Vehicle and Property Insurance Company_2500004_Xu v. Great American Insur/102/Ai estimate useful docs/FINAL_DRAFT_WITH_WITHOUT_REMOVAL_DEPRECIATION_REPORT_20250528_1.20250528193516823.1.PDF",
    "Aiestimate/Case File Exports/2550002_Rowlan v. Allstate Vehicle and Property Insurance Company_2500004_Xu v. Great American Insur/102/Ai estimate useful docs/608 Cutty Trl - Eagleview.PDF",
    "Aiestimate/Case File Exports/2550002_Rowlan v. Allstate Vehicle and Property Insurance Company_2500004_Xu v. Great American Insur/102/Ai estimate useful docs/Basic Damage Assessment For Primary Structure 03-24-2025-12-08-10pm (2).pdf",
    "Aiestimate/Case File Exports/2550002_Rowlan v. Allstate Vehicle and Property Insurance Company_2500004_Xu v. Great American Insur/102/Ai estimate useful docs/Engineering report.pdf",
   ])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (7213 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (9553 chars)
📝 Saved: outputs/Jul14/2500083/run5/output_code_lookup.txt
📝 Saved: outputs/Jul14/2500083/run5/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (12238 chars)
📝 Saved: outputs/Jul14/2500083/run5/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (8709 chars)
📝 Saved: outputs/Jul14/2500083/run5/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10094 chars)
📝 Saved: outputs/Jul14/2500083/run5/output_rebuttal.txt


In [ ]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul16/2500082/run3"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/101/ai estimate useful docs/2500082_Correspondence_Correspondence from IC_From_Claims, SLP_Burnett Estimate.pdf",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/101/ai estimate useful docs/6104_riviera_dr_north_richland_hills_tx_76180.pdf",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/101/ai estimate useful docs/8_7_23 - Donan Engineer Report.pdf",
    ])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (8612 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (15827 chars)
📝 Saved: outputs/Jul16/2500082/run3/output_code_lookup.txt
📝 Saved: outputs/Jul16/2500082/run3/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (14042 chars)
📝 Saved: outputs/Jul16/2500082/run3/output_scoping_logic.txt
🔹 Running ESTIMATE...


In [16]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul15/2500096/run7"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/118/ai estimate useful docs/USAA_REPORT.PDF.pdf",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/118/ai estimate useful docs/125 Hematite Lane, Jarrell, TX 76537 - RoofR.pdf",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/118/ai estimate useful docs/Kinsley Damage Photos (1).pdf",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/118/ai estimate useful docs/Merged Damage Photos Knisley.pdf",
    "Aiestimate/2500002_Ruiz Jr v. ASI Lloyds_2500005_Zimmerman v. Homesite Insurance Company_2500006_Zuniga v. Nati/118/ai estimate useful docs/Photos damaged shingles.pdf",
    ])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (8261 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (8329 chars)
📝 Saved: outputs/Jul15/2500096/run7/output_code_lookup.txt
📝 Saved: outputs/Jul15/2500096/run7/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (12652 chars)
📝 Saved: outputs/Jul15/2500096/run7/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (10947 chars)
📝 Saved: outputs/Jul15/2500096/run7/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10630 chars)
📝 Saved: outputs/Jul15/2500096/run7/output_rebuttal.txt
